# 4주차 실습: 한글 형태소 분석과 토큰화

**제출용** — TODO 1~7, 체크포인트 1~4의 답변과 실행 결과를 포함했습니다.

**주제:** Kiwi·KoNLPy(Okt)를 이용한 형태소 분석, 품사 태깅, 사용자 사전

## 학습 목표
1. 어절·형태소·토큰의 차이를 설명한다.
2. 공백 기반 토큰화와 형태소 분석 기반 토큰화를 비교한다.
3. Kiwi와 Okt로 품사 태깅을 수행하고 과업에 맞는 분석기를 판단한다.
4. 사용자 사전이 필요한 사례를 만들고 적용한다.

> **제출:** 모든 TODO를 작성·실행하고, 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보 포함 여부를 확인하세요.

In [1]:
# 실행 오류가 날 때만 아래 주석을 해제하세요.
# !pip -q install -U kiwipiepy konlpy

from importlib.metadata import version
import unicodedata

for package in ['kiwipiepy', 'konlpy', 'pandas']:
    print(f'{package}: {version(package)}')

import re
import pandas as pd
from kiwipiepy import Kiwi
from konlpy.tag import Okt

kiwi = Kiwi()
okt = Okt()

texts = [
    '충북대학교 자연어처리 수업이 정말 재미있어요!',
    'AI 챗봇의 답변 품질을 개선하고 싶어요.',
    '배터리가 오래가고 화면도 선명해서 만족합니다.',
    '결말은 아쉬웠지만 배우 연기는 최고였어요.',
    '새로 나온 갓생 챌린지 앱을 사용해 봤어요.'
]
df = pd.DataFrame({'text_id': range(1, len(texts)+1), 'text': texts})
display(df)

kiwipiepy: 0.23.2
konlpy: 0.6.0
pandas: 3.0.6


,text_id,text
0,1,충북대학교 자연어처리 수업이 정말 재미있어요!
1,2,AI 챗봇의 답변 품질을 개선하고 싶어요.
2,3,배터리가 오래가고 화면도 선명해서 만족합니다.
3,4,결말은 아쉬웠지만 배우 연기는 최고였어요.
4,5,새로 나온 갓생 챌린지 앱을 사용해 봤어요.


## 1. 어절·형태소·토큰
- **어절:** 띄어쓰기로 구분되는 단위
- **형태소:** 뜻 또는 문법 기능을 지닌 최소 단위
- **토큰:** 분석 목적에 맞게 모델 입력으로 정한 단위

예: `학생들이 도서관에서 공부한다.` → `학생/들/이`, `도서관/에서`, `공부/하/ㄴ다`처럼 분해할 수 있습니다.

In [2]:
sample = '학생들이 도서관에서 공부한다.'
print('원문:', sample)
print('공백 토큰:', sample.split())
print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))

# TODO 1 답안
kiwi_result = [(token.form, token.tag) for token in kiwi.tokenize(sample)]
print('Kiwi 형태소:', kiwi_result)

원문: 학생들이 도서관에서 공부한다.
공백 토큰: ['학생들이', '도서관에서', '공부한다.']
정규표현식 토큰: ['학생들이', '도서관에서', '공부한다']
Kiwi 형태소: [('학생', 'NNG'), ('들', 'XSN'), ('이', 'JKS'), ('도서관', 'NNG'), ('에서', 'JKB'), ('공부', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]


### 체크포인트 1
공백 토큰화로는 보이지 않는 문법·의미 정보 한 가지를 위 문장에서 찾아 1~2문장으로 설명하세요.

**답변**

공백 토큰화는 ‘학생들이’를 하나로 남기지만, Kiwi는 ‘학생/NNG’, ‘들/XSN’, ‘이/JKS’로 나누어 복수 접미사와 주격 조사를 보여 준다. 정규표현식 토큰화는 ‘공부한다.’의 마침표를 제거할 뿐 어절 내부를 분리하지 않으므로, 형태소 분석처럼 문법 기능을 구분하지는 못한다.

## 2. Kiwi로 형태소 분석과 품사 태깅
Kiwi는 현대 한국어 분석에 활용하기 좋은 형태소 분석기입니다. `token.form`은 형태, `token.tag`는 품사 태그입니다.

In [3]:
def kiwi_pos(text):
    return [(token.form, token.tag) for token in kiwi.tokenize(text)]

for text in df['text']:
    print(text)
    print(kiwi_pos(text))
    print()

# TODO 2 답안: 과제에서 지정한 N* 품사만 선택한다.
def kiwi_nouns(text):
    return [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith('N')]

print('첫 문장의 명사:', kiwi_nouns(df.loc[0, 'text']))
df['nouns'] = df['text'].apply(kiwi_nouns)
display(df[['text_id', 'text', 'nouns']])

충북대학교 자연어처리 수업이 정말 재미있어요!
[('충북대학교', 'NNP'), ('자연어 처리', 'NNP'), ('수업', 'NNG'), ('이', 'JKS'), ('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!', 'SF')]

AI 챗봇의 답변 품질을 개선하고 싶어요.
[('AI', 'SL'), ('챗봇', 'NNG'), ('의', 'JKG'), ('답변', 'NNG'), ('품질', 'NNG'), ('을', 'JKO'), ('개선', 'NNG'), ('하', 'XSV'), ('고', 'EC'), ('싶', 'VX'), ('어요', 'EF'), ('.', 'SF')]

배터리가 오래가고 화면도 선명해서 만족합니다.
[('배터리', 'NNG'), ('가', 'JKS'), ('오래', 'MAG'), ('가', 'VV'), ('고', 'EC'), ('화면', 'NNG'), ('도', 'JX'), ('선명', 'NNG'), ('하', 'XSA'), ('어서', 'EC'), ('만족', 'NNG'), ('하', 'XSA'), ('ᆸ니다', 'EF'), ('.', 'SF')]

결말은 아쉬웠지만 배우 연기는 최고였어요.
[('결말', 'NNG'), ('은', 'JX'), ('아쉽', 'VA-I'), ('었', 'EP'), ('지만', 'EC'), ('배우', 'NNG'), ('연기', 'NNG'), ('는', 'JX'), ('최고', 'NNG'), ('이', 'VCP'), ('었', 'EP'), ('어요', 'EF'), ('.', 'SF')]

새로 나온 갓생 챌린지 앱을 사용해 봤어요.
[('새로', 'MAG'), ('나오', 'VV'), ('ᆫ', 'ETM'), ('갓', 'MAG'), ('생', 'NNG'), ('챌린지', 'NNG'), ('앱', 'NNG'), ('을', 'JKO'), ('사용', 'NNG'), ('하', 'XSV'), ('어', 'EC'), ('보', 'VX'), ('었', 'EP'), ('어요', 'E

,text_id,text,nouns
0,1,충북대학교 자연어처리 수업이 정말 재미있어요!,"[충북대학교, 자연어 처리, 수업]"
1,2,AI 챗봇의 답변 품질을 개선하고 싶어요.,"[챗봇, 답변, 품질, 개선]"
2,3,배터리가 오래가고 화면도 선명해서 만족합니다.,"[배터리, 화면, 선명, 만족]"
3,4,결말은 아쉬웠지만 배우 연기는 최고였어요.,"[결말, 배우, 연기, 최고]"
4,5,새로 나온 갓생 챌린지 앱을 사용해 봤어요.,"[생, 챌린지, 앱, 사용]"


## 3. KoNLPy의 Okt로 형태소 분석
Okt는 소셜 텍스트·이모티콘·비표준 표현을 살펴볼 때 자주 사용합니다. 분석기마다 분리 방식과 품사 체계가 다를 수 있습니다.

In [4]:
compare_text = '오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ'
print('원문:', compare_text)
print('Kiwi:', kiwi_pos(compare_text))
print('Okt :', okt.pos(compare_text))

# TODO 3 답안: 같은 원문을 두 분석기의 기본 설정으로 비교한다.
my_sentence = '신조어와 줄임말도 분석할 수 있을까?'
print('\n비교 문장:', my_sentence)
print('Kiwi:', kiwi_pos(my_sentence))
print('Okt :', okt.pos(my_sentence))

원문: 오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ
Kiwi: [('오늘', 'MAG'), ('수업', 'NNG'), ('진짜', 'MAG'), ('재밌', 'VA'), ('었', 'EP'), ('는데', 'EC'), ('과제', 'NNG'), ('는', 'JX'), ('조금', 'MAG'), ('어렵', 'VA-I'), ('네요', 'EF'), ('ㅠㅠ', 'SW')]
Okt : [('오늘', 'Noun'), ('수업', 'Noun'), ('진짜', 'Noun'), ('재밌었는데', 'Adjective'), ('과제', 'Noun'), ('는', 'Josa'), ('조금', 'Noun'), ('어렵네요', 'Adjective'), ('ㅠㅠ', 'KoreanParticle')]

비교 문장: 신조어와 줄임말도 분석할 수 있을까?
Kiwi: [('신조어', 'NNG'), ('와', 'JC'), ('줄임', 'NNG'), ('말', 'NNG'), ('도', 'JX'), ('분석', 'NNG'), ('하', 'XSV'), ('ᆯ', 'ETM'), ('수', 'NNB'), ('있', 'VA'), ('을까', 'EF'), ('?', 'SF')]
Okt : [('신조어', 'Noun'), ('와', 'Josa'), ('줄임말도', 'Verb'), ('분석', 'Noun'), ('할', 'Verb'), ('수', 'Noun'), ('있을까', 'Adjective'), ('?', 'Punctuation')]


### 체크포인트 2
Kiwi와 Okt 결과에서 토큰 분리 또는 품사 태깅이 다른 사례를 하나 기록하세요. 어느 분석기가 “더 좋다”가 아니라, 어떤 **데이터/과업**에서 더 편리할지 설명하세요.

**답변**

첫 문장의 ‘재밌었는데’를 Kiwi는 ‘재밌/VA’, ‘었/EP’, ‘는데/EC’로 나누지만, Okt는 ‘재밌었는데/Adjective’로 남긴다. 어간·시제·연결 어미를 따로 이용하는 감성 분석에는 Kiwi가 편리하고, 활용된 표현을 묶어 소셜 텍스트의 어휘를 탐색할 때는 Okt가 편리할 수 있다.

추가 문장의 ‘줄임말도’는 Kiwi에서 ‘줄임/NNG’, ‘말/NNG’, ‘도/JX’로, Okt에서 ‘줄임말도/Verb’로 나타났다. 문맥상 명사 ‘줄임말’에 보조사 ‘도’가 붙은 표현이므로 이 예에서는 Okt의 품사 판단이 어색하며, Kiwi도 ‘줄임말’을 한 단위로 보존하지는 않았다. 따라서 신조어·복합명사 키워드를 추출할 때는 분석 결과를 원문과 대조하고 사용자 사전도 검토해야 한다.

## 4. 과업에 따른 품사 선택
키워드 추출은 명사 중심 처리가 유용할 수 있습니다. 반면 감성 분석에서는 형용사·동사·부정 표현·강조 부사도 중요한 신호가 됩니다.

In [5]:
review = '배송은 조금 느렸지만 제품 디자인이 매우 예쁘고 만족스러워요.'
print('전체 분석:', kiwi_pos(review))

# TODO 4 답안
def select_pos(text):
    nouns = []
    predicates = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith('N'):
            nouns.append(token.form)
        elif token.tag.startswith('V'):
            predicates.append(token.form)
    return nouns, predicates

nouns, predicates = select_pos(review)
print('명사:', nouns)
print('용언:', predicates)

전체 분석: [('배송', 'NNG'), ('은', 'JX'), ('조금', 'MAG'), ('느리', 'VA'), ('었', 'EP'), ('지만', 'EC'), ('제품', 'NNG'), ('디자인', 'NNG'), ('이', 'JKS'), ('매우', 'MAG'), ('예쁘', 'VA'), ('고', 'EC'), ('만족', 'NNG'), ('스럽', 'XSA-I'), ('어요', 'EF'), ('.', 'SF')]
명사: ['배송', '제품', '디자인', '만족']
용언: ['느리', '예쁘']


### 체크포인트 3
`좋지 않아요`, `매우 예쁘고` 같은 표현을 생각해 보세요. 명사만 남기면 감성 분석에서 어떤 정보가 사라질까요?

**답변**

명사만 남기면 ‘좋지 않아요’의 평가 어간 ‘좋’과 부정 표현 ‘않’, ‘매우 예쁘고’의 강도 부사 ‘매우’와 평가 어간 ‘예쁘’가 사라진다. 명사 중심 처리는 ‘배송’, ‘제품’, ‘디자인’처럼 평가 대상을 찾는 데 유용하지만, 대상에 대한 긍정·부정과 강도까지 판단하기에는 부족하다.

위 실행에서 ‘만족스러워요’는 ‘만족/NNG + 스럽/XSA-I + 어요/EF’로 나뉘므로 V*만 선택해도 완전한 평가 표현을 얻지는 못한다. 감성용 정책에서는 용언뿐 아니라 부사, 부정 표현과 관련 어미·접미사도 함께 보존할 필요가 있다.

## 5. 사용자 사전: 왜 필요한가
새 서비스명, 과목명, 브랜드명, 전문 용어는 일반 사전에 없거나 원하지 않는 방식으로 분리될 수 있습니다. 사용자 사전은 **도메인 용어를 원하는 품사와 단위로 분석**하도록 돕습니다.

아래 코드는 Kiwi 버전에 따라 사용자 단어 추가 방식이 다를 수 있으므로, 실행 환경의 오류 메시지와 라이브러리 문서를 함께 확인하세요.

In [6]:
# TODO 5 답안: 셀을 다시 실행해도 추가 전 결과를 비교할 수 있도록 초기화한다.
kiwi = Kiwi()
domain_text = '충북대학교에서 자연어처리 프로젝트를 진행한다.'
domain_before = kiwi_pos(domain_text)
print('추가 전:', domain_before)
for word in ['자연어처리', '충북대학교']:
    added = kiwi.add_user_word(word, 'NNP')
    print(f'{word} 등록 여부:', added)
domain_after = kiwi_pos(domain_text)
print('추가 후:', domain_after)

# TODO 6 답안: 맥북 관심 분야와 연결한 제품명·서비스명
interest_text = '맥스튜디오에서 애플인텔리전스 기능을 테스트했다.'
interest_before = kiwi_pos(interest_text)
print('\n관심 분야 문장:', interest_text)
print('추가 전:', interest_before)
for word in ['맥스튜디오', '애플인텔리전스']:
    # 도메인 용어가 한 단위의 고유명사로 선택되도록 가중치를 부여한다.
    kiwi.add_user_word(word, 'NNP', score=5.0)
interest_after = kiwi_pos(interest_text)
print('추가 후:', interest_after)

display(pd.DataFrame([
    {'문장': domain_text, '추가 전': domain_before, '추가 후': domain_after},
    {'문장': interest_text, '추가 전': interest_before, '추가 후': interest_after},
]))

추가 전: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]
자연어처리 등록 여부: True
충북대학교 등록 여부: False
추가 후: [('충북대학교', 'NNP'), ('에서', 'JKB'), ('자연어 처리', 'NNP'), ('프로젝트', 'NNG'), ('를', 'JKO'), ('진행', 'NNG'), ('하', 'XSV'), ('ᆫ다', 'EF'), ('.', 'SF')]

관심 분야 문장: 맥스튜디오에서 애플인텔리전스 기능을 테스트했다.
추가 전: [('맥', 'NNP'), ('스튜디오', 'NNG'), ('에서', 'JKB'), ('애플', 'NNP'), ('인텔리전스', 'NNP'), ('기능', 'NNG'), ('을', 'JKO'), ('테스트', 'NNG'), ('하', 'XSV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
추가 후: [('맥스튜디오', 'NNP'), ('에서', 'JKB'), ('애플인텔리전스', 'NNP'), ('기능', 'NNG'), ('을', 'JKO'), ('테스트', 'NNG'), ('하', 'XSV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]


,문장,추가 전,추가 후
0,충북대학교에서 자연어처리 프로젝트를 진행한다.,"[(충북대학교, NNP), (에서, JKB), (자연어 처리, NNP), (프로젝트...","[(충북대학교, NNP), (에서, JKB), (자연어 처리, NNP), (프로젝트..."
1,맥스튜디오에서 애플인텔리전스 기능을 테스트했다.,"[(맥, NNP), (스튜디오, NNG), (에서, JKB), (애플, NNP), ...","[(맥스튜디오, NNP), (에서, JKB), (애플인텔리전스, NNP), (기능,..."


### 사용자 사전 적용 결과 해석

기본 예문의 ‘충북대학교’는 추가 전부터 NNP로 인식되어 등록 결과가 `False`이고 출력도 동일했다. ‘자연어처리’는 등록되었지만 이번 버전에서는 추가 후에도 ‘자연어 처리/NNP’로 출력되었다. 사용자 단어 등록 여부와 실제 출력 변화는 구분해서 확인해야 한다.

관심 분야 예문에서는 ‘맥/NNP + 스튜디오/NNG’가 ‘맥스튜디오/NNP’로, ‘애플/NNP + 인텔리전스/NNP’가 ‘애플인텔리전스/NNP’로 바뀌었다. 두 이름을 하나의 고유명사로 보존하면 제품·서비스별 검색이나 키워드 집계에 유용하다. 다만 `score=5.0`은 이번 예문에 적용한 가중치이므로, 다른 문맥에서도 적절한지는 따로 확인해야 한다.

## 6. 미니 과제: 토큰화 정책 설계
다음 목표 중 하나를 선택해 토큰화 정책을 설계하세요.
1. 강의평 키워드 추출
2. 영화 리뷰 감성 분석
3. 학교 민원 자동 분류

아래 함수에 분석기, 보존할 품사, 제거할 토큰의 기준을 작성하세요.

In [7]:
# TODO 7 답안: 영화 리뷰 감성 분석용 토큰화
# 이모지 일부, 한글 감정 문자, 반복 문장부호는 분리 전에 보존한다.
emotion_pattern = re.compile(
    r'([ㅋㅎㅠㅜ]{2,}|[!?]{2,}|\.{3,}|[\U0001F300-\U0001FAFF\u2600-\u27BF]\ufe0f?)'
)

def tokenize_for_task(text):
    """명사·용언·수식언·어미·접사를 남기고 감정 표현의 순서를 보존한다."""
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    tokens = []
    for part in emotion_pattern.split(text):
        if not part.strip():
            continue
        if emotion_pattern.fullmatch(part):
            tokens.append(part)
            continue
        for token in kiwi.tokenize(part):
            tag = token.tag.split('-')[0]
            if (tag.startswith(('N', 'V', 'M', 'E'))
                    or tag in {'XPN', 'XSN', 'XSV', 'XSA', 'SL', 'SN'}):
                tokens.append(token.form)
    return tokens

for text in df['text']:
    print(text, '->', tokenize_for_task(text))

# 부정·강조·이모지·반복 문자·URL 처리 결과를 확인한다.
policy_examples = [
    '영화가 좋지 않아요.',
    '영상이 매우 예쁘고 음악도 좋아요!!! 😊',
    '정말 안 무서워요 ㅋㅋㅋㅋ',
    '결말은 별로예요 ㅠㅠ https://example.com/review',
]
print('\n감성 표현 적용 예시')
for text in policy_examples:
    print(text, '->', tokenize_for_task(text))

충북대학교 자연어처리 수업이 정말 재미있어요! -> ['충북대학교', '자연어 처리', '수업', '정말', '재미있', '어요']
AI 챗봇의 답변 품질을 개선하고 싶어요. -> ['AI', '챗봇', '답변', '품질', '개선', '하', '고', '싶', '어요']
배터리가 오래가고 화면도 선명해서 만족합니다. -> ['배터리', '오래', '가', '고', '화면', '선명', '하', '어서', '만족', '하', 'ᆸ니다']
결말은 아쉬웠지만 배우 연기는 최고였어요. -> ['결말', '아쉽', '었', '지만', '배우', '연기', '최고', '이', '었', '어요']
새로 나온 갓생 챌린지 앱을 사용해 봤어요. -> ['새로', '나오', 'ᆫ', '갓', '생', '챌린지', '앱', '사용', '하', '어', '보', '었', '어요']

감성 표현 적용 예시
영화가 좋지 않아요. -> ['영화', '좋', '지', '않', '어요']
영상이 매우 예쁘고 음악도 좋아요!!! 😊 -> ['영상', '매우', '예쁘', '고', '음악', '좋', '어요', '!!!', '😊']
정말 안 무서워요 ㅋㅋㅋㅋ -> ['정말', '안', '무섭', '어요', 'ㅋㅋㅋㅋ']
결말은 별로예요 ㅠㅠ https://example.com/review -> ['결말', '별로', '이', '예요', 'ㅠㅠ']


### 체크포인트 4
선택한 과업, 분석기 선택 이유, 보존/제거 규칙, 예상되는 오류 사례를 각각 1문장 이상 작성하세요.

**답변**

- **선택 과업:** 영화 리뷰 감성 분석을 선택했으며, 평가 대상뿐 아니라 긍정·부정 표현과 강조의 단서를 남기는 것을 목표로 했다.
- **분석기 선택 이유:** Kiwi는 ‘좋지 않아요’를 어간·부정 보조 용언·어미로 나누어 보여 주므로, 부정 표현을 보존하는 품사 규칙을 작성하기 편리하다.
- **보존 규칙:** 명사(N*), 용언(V*), 수식언(M*), 어미(E*)와 접사(XPN·XSN·XSV·XSA), 외국어(SL)·숫자(SN)를 남긴다. ‘않’, ‘안’, ‘매우’가 제거되지 않도록 별도의 불용어 목록은 사용하지 않으며, 이모지 일부와 ‘ㅋㅋㅋㅋ’, ‘ㅠㅠ’, ‘!!!’ 같은 감정 표현은 원래 형태와 순서로 보존한다.
- **제거 규칙:** URL은 공백으로 바꾸고 조사와 일반 기호는 제거한다. 단, 반복 느낌표·물음표·세 개 이상의 마침표는 강조나 여운의 단서로 남기며, 단어 내부의 반복은 일괄 축약하지 않는다.
- **예상 오류:** ‘정말 잘 만들었네요, 잠이 솔솔 와요’ 같은 반어 표현은 토큰만으로 정확히 판단하기 어렵고, 조사를 제거하면 ‘연기는 좋지만 내용은 별로’처럼 평가 대상의 대비가 약해질 수 있다. 이모지 처리는 일부 유니코드 범위만 다루므로 결합 이모지를 완전하게 보존하지 못할 수 있으며, 신조어나 오탈자도 추가 검토가 필요하다. 이 함수는 토큰화 정책이며 감성 분류 모델의 정확도를 검증한 결과는 아니다.

## AI 활용 및 실행 기록

**미니 과제에서 AI를 적극 활용** 

품사 선택 구현: 명사·용언·부사·어미를 남기고 조사와 일반 기호를 제거하는 조건문 작성
예외 처리: URL 제거, ㅠㅠ·ㅋㅋㅋㅋ·!!!·이모지 보존을 위한 정규표현식 작성
검증 예문 구성: “좋지 않아요”의 부정 표현, “매우 예쁘고”의 강조 표현 등이 실제로 남는지 확인


## 프로젝트 검증 수행

1. 전체 실행: 새 커널에서 코드 셀 7개를 위에서 아래로 실행했고, 오류 없이 실행 결과를 저장했습니다.
2. 분석 결과 대조: Kiwi·Okt의 실제 토큰과 품사 출력을 확인하고, 체크포인트 설명이 출력과 일치하는지 검토했습니다.
3. 사용자 사전 확인: 맥스튜디오, 애플인텔리전스가 추가 전에는 분리되고 추가 후에는 각각 하나의 고유명사로 분석되는 것을 확인했습니다.
4. 미니 과제 동작 확인: 좋·지·않, 매우, 안, 😊, ㅋㅋㅋㅋ, ㅠㅠ가 보존되고 URL이 제거되는 것을 예문 출력에서 확인했습니다.
5. 파일 완성도 확인: 노트북 형식, 실행 순서, 오류 출력 유무, 체크포인트 4개의 답변과 제출 체크리스트를 검사했습니다.
다만 별도의 단위 테스트나 실제 리뷰 데이터 전체에 대한 정확도 평가는 하지 않았습니다.


## 제출 체크리스트
- [x] TODO 1~7을 작성하고 실행했다.
- [x] 공백·정규표현식·형태소 기반 토큰화를 비교했다.
- [x] Kiwi와 Okt의 결과 차이 및 과업 적합성을 설명했다.
- [x] 명사 중심 처리의 장점과 한계를 설명했다.
- [x] 사용자 사전에 고유명사 2개 이상을 추가하고 전후 결과를 비교했다.
- [x] 선택 과업에 대한 토큰화 정책과 오류 사례를 기록했다.
- [x] 외부 AI/자료 사용 시 범위와 검증 과정을 기록했다.

## 다음 주차 예고
전처리된 토큰을 Bag-of-Words, TF-IDF, n-gram으로 수치화하여 문서 비교와 분류 모델의 입력 특징을 구성합니다.